
# Minimal HF Datasets: Collate → Clean → Deduplicate → Export

This notebook uses **Hugging Face Datasets** to:
- Load multiple CSV files into a single `Dataset`
- Clean text (strip + collapse spaces)
- Remove rows where `len(text) ≤ 3`
- Deduplicate by (`text`, `label`)
- Save the result to **CSV** (and optionally **Parquet**)

> This is the **simplest, fastest** approach for large text datasets—no pandas merging needed.


In [ ]:

# If datasets is not installed, uncomment:
# !pip install -q datasets pyarrow


In [ ]:

from datasets import load_dataset, Dataset
import os

# ====== EDIT THESE ======
CSV_PATH = [
    # Put your CSVs here (absolute or relative paths)
    "file1.csv",
    "file2.csv",
    "file3.csv",
]

OUT_CSV = "combined_clean.csv"      # output CSV file
OUT_PARQUET = "combined_clean.parquet"  # optional Parquet

# If your CSVs have headers with exact columns "text" and "label", keep this True.
HAS_HEADER = True

# If your CSVs DO NOT have a header and the first two columns are "text", "label", set this False:
# HAS_HEADER = False


In [ ]:

# ---- Load CSVs into a single Dataset ----
if HAS_HEADER:
    ds = load_dataset("csv", data_files=CSV_PATH, split="train")
else:
    ds = load_dataset("csv", data_files=CSV_PATH, split="train",
                      column_names=["text", "label"])  # first two cols = text,label

print(ds)
print(ds.features)
print(f"Rows loaded: {len(ds):,}")


In [ ]:

# ---- Clean & Deduplicate ----

def clean_map(ex):
    t = str(ex["text"]).strip()
    t = " ".join(t.split())  # collapse multiple spaces/newlines
    ex["text"] = t
    return ex

# Use multiple processes for speed (adjust num_proc as desired)
num_proc = 4

ds = ds.map(clean_map, num_proc=num_proc, desc="Cleaning text")
ds = ds.filter(lambda ex: len(ex["text"]) > 3, num_proc=num_proc, desc="Dropping short (<=3)")

# Dedup with pandas
df = ds.to_pandas()
df = df.dropna(subset=["text", "label"]).drop_duplicates(
    subset=["text", "label"]).reset_index(drop=True)

# Convert back
ds = Dataset.from_pandas(df, preserve_index=False)
print(ds)

print(ds)
print(f"Rows after clean+dedup: {len(ds):,}")

In [ ]:

# ---- Save to CSV/Parquet ----
ds.to_csv(OUT_CSV)
print(f"✅ Wrote CSV: {os.path.abspath(OUT_CSV)}")

# Optional Parquet (comment out if not needed)
try:
    ds.to_parquet(OUT_PARQUET)
    print(f"✅ Wrote Parquet: {os.path.abspath(OUT_PARQUET)}")
except Exception as e:
    print("Parquet write skipped or failed:", e)


In [ ]:

# ---- Quick sanity checks ----
# Peek a few rows
print(ds.select(range(min(5, len(ds)))))

# Label distribution (if labels are strings)
try:
    from collections import Counter
    cnt = Counter(ds["label"])
    print("Label counts (top 10):", cnt.most_common(10))
except Exception as e:
    print("Could not compute label counts:", e)
